In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/serie_a_last_5_seasons_matches.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

df.head()


,date,home_team,away_team,home_goals,away_goals
0,2020-09-19,Hellas Verona,Roma,3.0,0.0
1,2020-09-19,Fiorentina,Torino,1.0,0.0
2,2020-09-20,Genoa,Crotone,4.0,1.0
3,2020-09-20,Sassuolo,Cagliari,1.0,1.0
4,2020-09-20,Juventus,Sampdoria,3.0,0.0


In [2]:
# Create team-level rows (one row per team per match)

home = df[["date", "home_team", "away_team", "home_goals", "away_goals"]].copy()
home["team"] = home["home_team"]
home["opponent"] = home["away_team"]
home["venue"] = "Home"
home["gf"] = home["home_goals"]
home["ga"] = home["away_goals"]

away = df[["date", "home_team", "away_team", "home_goals", "away_goals"]].copy()
away["team"] = away["away_team"]
away["opponent"] = away["home_team"]
away["venue"] = "Away"
away["gf"] = away["away_goals"]
away["ga"] = away["home_goals"]

team_rows = pd.concat([home, away], ignore_index=True)

team_rows = team_rows[["date", "team", "opponent", "venue", "gf", "ga"]]
team_rows.head()


,date,team,opponent,venue,gf,ga
0,2020-09-19,Hellas Verona,Roma,Home,3.0,0.0
1,2020-09-19,Fiorentina,Torino,Home,1.0,0.0
2,2020-09-20,Genoa,Crotone,Home,4.0,1.0
3,2020-09-20,Sassuolo,Cagliari,Home,1.0,1.0
4,2020-09-20,Juventus,Sampdoria,Home,3.0,0.0


In [3]:
ROLLING_N = 5

team_rows = team_rows.sort_values(["team", "date"]).copy()

team_rows["avg_goals_scored"] = (
    team_rows.groupby("team")["gf"]
    .shift(1)
    .rolling(ROLLING_N, min_periods=ROLLING_N)
    .mean()
    .reset_index(level=0, drop=True)
)

team_rows["avg_goals_conceded"] = (
    team_rows.groupby("team")["ga"]
    .shift(1)
    .rolling(ROLLING_N, min_periods=ROLLING_N)
    .mean()
    .reset_index(level=0, drop=True)
)

team_rows.head(10)


,date,team,opponent,venue,gf,ga,avg_goals_scored,avg_goals_conceded
1813,2020-09-26,Atalanta,Torino,Away,4.0,2.0,1.6,0.8
1824,2020-09-30,Atalanta,Lazio,Away,4.0,1.0,2.0,1.2
26,2020-10-04,Atalanta,Cagliari,Home,5.0,2.0,3.0,0.8
1834,2020-10-17,Atalanta,Napoli,Away,1.0,4.0,1.4,0.8
39,2020-10-24,Atalanta,Sampdoria,Home,1.0,3.0,1.6,1.2
1853,2020-10-31,Atalanta,Crotone,Away,2.0,1.0,1.6,0.8
63,2020-11-08,Atalanta,Internazionale,Home,1.0,1.0,1.2,1.0
1874,2020-11-21,Atalanta,Spezia,Away,0.0,0.0,1.4,1.2
75,2020-11-28,Atalanta,Hellas Verona,Home,0.0,2.0,1.2,1.0
102,2020-12-13,Atalanta,Fiorentina,Home,3.0,0.0,1.0,1.2


In [4]:
# Separate home and away features

home_features = team_rows[team_rows["venue"] == "Home"][
    ["date", "team", "avg_goals_scored", "avg_goals_conceded"]
].rename(
    columns={
        "team": "home_team",
        "avg_goals_scored": "home_avg_scored",
        "avg_goals_conceded": "home_avg_conceded",
    }
)

away_features = team_rows[team_rows["venue"] == "Away"][
    ["date", "team", "avg_goals_scored", "avg_goals_conceded"]
].rename(
    columns={
        "team": "away_team",
        "avg_goals_scored": "away_avg_scored",
        "avg_goals_conceded": "away_avg_conceded",
    }
)

# Merge back to match-level
features = df.merge(home_features, on=["date", "home_team"], how="left")
features = features.merge(away_features, on=["date", "away_team"], how="left")

features.head()


,date,home_team,away_team,home_goals,away_goals,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded
0,2020-09-19,Hellas Verona,Roma,3.0,0.0,NaN,NaN,1.6,1.4
1,2020-09-19,Fiorentina,Torino,1.0,0.0,NaN,NaN,1.8,1.2
2,2020-09-20,Genoa,Crotone,4.0,1.0,NaN,NaN,2.6,1.0
3,2020-09-20,Sassuolo,Cagliari,1.0,1.0,NaN,NaN,2.4,0.8
4,2020-09-20,Juventus,Sampdoria,3.0,0.0,NaN,NaN,2.2,0.6


In [5]:
features = features.dropna().copy()
features["home_advantage"] = 1

features.shape


(1666, 10)

In [6]:
features.to_csv("../data/processed/features.csv", index=False)
